# Flash Drum and Distillation Column

Simulating two fundamental separation processes: an isothermal binary flash drum and a multi-tray distillation column built from individual `DistillationTray` blocks wired in series.

## Part 1: Isothermal Flash Drum

A flash drum separates a liquid feed into vapor and liquid streams using vapor-liquid equilibrium (VLE). The drum uses Raoult's law with Antoine correlations to compute K-values:

$$K_i = \frac{P^\text{sat}_i(T)}{P}$$

The Rachford-Rice equation determines the vapor fraction $\beta$, from which the vapor ($y_i$) and liquid ($x_i$) compositions follow.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('../pathsim_docs.mplstyle')

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope

from pathsim_chem.process import FlashDrum, DistillationTray

Configure a flash drum for a benzene-toluene mixture (default Antoine coefficients). Feed an equimolar mixture at 1 atm and sweep temperature from 340 K to 400 K. This range covers the bubble point (~365 K) and dew point (~380 K) of the mixture, so we can observe the transition from all-liquid to two-phase to all-vapor.

In [ ]:
flash = FlashDrum(holdup=100.0)  # default benzene/toluene Antoine params

# Feed: 10 mol/s, equimolar (z1 = 0.5), 1 atm
src_f = Source(lambda t: 10.0)
src_z = Source(lambda t: 0.5)
src_t = Source(lambda t: 340.0 + t * 0.5)   # ramp 340 -> 400 K
src_p = Source(lambda t: 101325.0)

sco = Scope(labels=["V_rate", "L_rate", "y_1 (benzene)", "x_1 (benzene)"])

sim = Simulation(
    [src_f, src_z, src_t, src_p, flash, sco],
    [
        Connection(src_f, flash),          # F
        Connection(src_z, flash[1]),       # z_1
        Connection(src_t, flash[2]),       # T
        Connection(src_p, flash[3]),       # P
        Connection(flash, sco),            # V_rate
        Connection(flash[1], sco[1]),      # L_rate
        Connection(flash[2], sco[2]),      # y_1
        Connection(flash[3], sco[3]),      # x_1
    ],
    dt=0.5,
    log=True,
)

sim.run(120)

In [ ]:
time, [V_rate, L_rate, y1, x1] = sco.read()
T_sweep = 340.0 + time * 0.5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(T_sweep, V_rate, label="Vapor rate")
ax1.plot(T_sweep, L_rate, label="Liquid rate")
ax1.set_xlabel("Temperature [K]")
ax1.set_ylabel("Flow rate [mol/s]")
ax1.set_title("Flash Drum Flow Rates")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(T_sweep, y1, label=r"$y_1$ (vapor)")
ax2.plot(T_sweep, x1, label=r"$x_1$ (liquid)")
ax2.axhline(0.5, color="gray", ls="--", alpha=0.4, label="Feed")
ax2.set_xlabel("Temperature [K]")
ax2.set_ylabel("Mole fraction (benzene)")
ax2.set_title("VLE Compositions")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

As temperature increases, more liquid vaporizes (higher vapor rate). The vapor is enriched in the lighter component (benzene), while the liquid becomes richer in toluene — the classic VLE separation.

## Part 2: Distillation Column (5 Trays)

A distillation column is built by wiring multiple `DistillationTray` blocks in series. Each tray enforces vapor-liquid equilibrium with a constant relative volatility $\alpha$:

$$y = \frac{\alpha\, x}{1 + (\alpha - 1)\, x}$$

Under constant molar overflow (CMO), liquid flows down and vapor flows up with constant rates $L$ and $V$. We feed the column at the middle tray.

In [ ]:
# Column parameters
N_trays = 5
alpha = 2.5       # relative volatility
L = 5.0           # liquid flow rate [mol/s]
V = 5.0           # vapor flow rate [mol/s]

# Create tray blocks (all start at x = 0.5)
trays = [DistillationTray(M=1.0, alpha=alpha, x0=0.5) for _ in range(N_trays)]

# Liquid feed from condenser (enters tray 0 from above)
src_l = Source(lambda t: L)
src_xtop = Source(lambda t: 0.95)   # reflux composition

# Vapor feed from reboiler (enters tray N-1 from below)
src_v = Source(lambda t: V)
src_ybot = Source(lambda t: 0.05)   # reboiler vapor

# Record composition on each tray
sco = Scope(labels=[f"Tray {i+1}" for i in range(N_trays)])

Wire the trays: liquid cascades downward (tray $i$ liquid output $\to$ tray $i+1$ liquid input), vapor rises upward (tray $i$ vapor output $\to$ tray $i-1$ vapor input). External feeds enter the top and bottom.

In [ ]:
connections = []

# Top tray (0): liquid from reflux
connections.append(Connection(src_l, trays[0]))        # L_in
connections.append(Connection(src_xtop, trays[0][1]))  # x_in

# Bottom tray (N-1): vapor from reboiler
connections.append(Connection(src_v, trays[-1][2]))    # V_in
connections.append(Connection(src_ybot, trays[-1][3])) # y_in

# Inter-tray connections
for i in range(N_trays - 1):
    # Liquid flows down: tray i -> tray i+1
    connections.append(Connection(trays[i], trays[i+1]))       # L_out -> L_in
    connections.append(Connection(trays[i][1], trays[i+1][1])) # x_out -> x_in

    # Vapor flows up: tray i+1 -> tray i
    connections.append(Connection(trays[i+1][2], trays[i][2])) # V_out -> V_in
    connections.append(Connection(trays[i+1][3], trays[i][3])) # y_out -> y_in

# Connect each tray's liquid composition to scope
for i, tray in enumerate(trays):
    connections.append(Connection(tray[1], sco[i]))  # x_out

sim = Simulation(
    [src_l, src_xtop, src_v, src_ybot, *trays, sco],
    connections,
    dt=0.05,
    log=True,
)

sim.run(30)

In [ ]:
time, signals = sco.read()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Dynamic tray composition evolution
for i in range(N_trays):
    ax1.plot(time, signals[i], label=f"Tray {i+1}")
ax1.set_xlabel("Time [s]")
ax1.set_ylabel(r"$x$ (light component)")
ax1.set_title("Tray Compositions Over Time")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Steady-state composition profile (from scope data)
x_profile = [signals[i][-1] for i in range(N_trays)]
tray_nums = list(range(1, N_trays + 1))

ax2.plot(tray_nums, x_profile, "o-", markersize=8)
ax2.set_xlabel("Tray number (top to bottom)")
ax2.set_ylabel(r"$x$ (light component)")
ax2.set_title("Composition Profile (Steady State)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The column separates the light and heavy components across its trays. The top tray is enriched in the light component (high $x$) while the bottom tray is depleted. The steady-state composition profile shows the characteristic staircase that a McCabe-Thiele diagram would predict for this relative volatility.